# 13 · pT-binned comparison, overlap check and paired uncertainties

The independent review found that the inclusive tie with the CMS likelihood (jetQGl) is driven by pT composition:
models trained on 400 inclusive jets gain ~0.02 AUC because the quark fraction rises with pT, while jetQGl is built in pT bins.

This notebook, on the same confirmation file, with the same frozen protocols as notebooks 10–11:
1. checks that the development and confirmation files share no events,
2. re-fits the key models and **saves per-jet scores**,
3. reports inclusive, per-pT-bin and **pT-reweighted** AUCs (gluon jets reweighted to the quark pT spectrum, so pT carries no information),
4. adds the single-feature reference **QG_axis2 alone** (no training),
5. gives **paired bootstrap** 68% intervals for AUC differences (resampling confirmation jets, averaging over the 5 seeds).

In [ ]:
import sys
sys.path.insert(0, "src")

import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

from jet_data import (load_jets, select_quark_gluon, feature_matrix, event_level_split, balanced_subsample,
                      leading_constituents, BOOKKEEPING, PF_KEYS)
from kernels import (KERNELS, preprocess, onep1q_states, onep1q_kernel, angle_kernel, shot_noise)
from figure_style import set_style, save, COLORS

set_style()
SEED = 42
SEEDS = [0, 1, 2, 3, 4]
N_PER_CLASS = 200
N_CONST = 10
QGL_INPUTS = ("QG_mult", "QG_ptD", "QG_axis2")
C_GRID = np.logspace(-2, 6, 9)                       # 0.01 ... 1e6 (concentrated kernels may need large C)
F_GRID = np.round(np.logspace(0, 2.5, 12), 2)        # 1 ... 316
SHOTS = [100, 1000]
MAX_ITER = 200_000                                  # caps solver time at very large C; unconverged fits just score worse in CV
DEV_PATH = Path("Data/JetNtuple_RunIISummer16_13TeV_MC_1.h5")
CONFIRM_PATH = Path("Data/JetNtuple_RunIISummer16_13TeV_MC_10.h5")   # second, never-used file of record 12100
RESULTS = Path("results")
assert CONFIRM_PATH.exists(), f"confirmation file not found: {CONFIRM_PATH}"

COLUMNS = QGL_INPUTS + ("jetPt", "jetQGl")
def load(path):
    jets = select_quark_gluon(load_jets(path=path, features=COLUMNS, extra=BOOKKEEPING + PF_KEYS), require_loose_id=True)
    z, de, dp, _ = leading_constituents(jets, n_max=N_CONST)
    return {"y": jets["label"], "X": feature_matrix(jets, features=QGL_INPUTS), "pt": np.asarray(jets["jetPt"], float),
            "qgl": np.asarray(jets["jetQGl"], float), "z": z, "de": de, "dp": dp, "dr": np.sqrt(de**2 + dp**2), "jets": jets}

dev = load(DEV_PATH)
conf = load(CONFIRM_PATH)
train, test = event_level_split(dev["jets"], test_fraction=0.3, seed=SEED)
dev_test = np.flatnonzero(test)
print(f"development file: {len(dev['y'])} jets (train {int(train.sum())}, test {len(dev_test)})")
print(f"confirmation file: {len(conf['y'])} jets, quark fraction {conf['y'].mean():.3f}")

In [ ]:
# name -> (family, how to build the kernel)
QUANTUM = {
    "amplitude, raw":            ("amplitude", "qgl", "raw"),
    "amplitude, standardised":   ("amplitude", "qgl", "standard"),
    "amplitude, min-max":        ("amplitude", "qgl", "minmax"),
    "angle, robust + YJ":        ("angle", "qgl", "robust_yj"),
    "angle, min-max":            ("angle", "qgl", "minmax"),
    "1P1Q (original)":           ("1P1Q", "const", "original"),
    "1P1Q rotation-invariant":   ("1P1Q", "const", "invariant"),
}

def kernel_builder(name, train_idx, f=None):
    family, inputs, variant = QUANTUM[name]
    if inputs == "qgl":
        Xtr = dev["X"][train_idx]
        def transform(X):
            return preprocess(variant, Xtr, X)[1]
        A = transform(Xtr)
        k = KERNELS[family]
        return lambda D, rows: k(transform(D["X"][rows]), A), k(A, A)
    if variant == "original":
        st = lambda D, rows: onep1q_states(D["z"][rows], D["de"][rows], D["dp"][rows], f)
        SA = st(dev, train_idx)
        return lambda D, rows: onep1q_kernel(st(D, rows), SA), onep1q_kernel(SA, SA)
    ang = lambda D, rows: f * D["z"][rows] * D["dr"][rows]
    A = ang(dev, train_idx)
    return lambda D, rows: angle_kernel(ang(D, rows), A), angle_kernel(A, A)

def cv_auc(K, labels, folds, C):
    out = []
    for tr, va in folds:
        svc = SVC(kernel="precomputed", C=C, max_iter=MAX_ITER).fit(K[np.ix_(tr, tr)], labels[tr])
        out.append(roc_auc_score(labels[va], svc.decision_function(K[np.ix_(va, tr)])))
    return np.mean(out)
print(list(QUANTUM))

## 1. Event overlap between the development and confirmation files

In [ ]:
def event_keys(D):
    j = D["jets"]
    return set(zip(np.asarray(j["run"]).tolist(), np.asarray(j["lumi"]).tolist(), np.asarray(j["event"]).tolist()))
overlap = event_keys(dev) & event_keys(conf)
print(f"events in dev file: {len(event_keys(dev))}, in confirmation file: {len(event_keys(conf))}, shared: {len(overlap)}")

## 2. Re-fit key models and save per-jet scores (about 5 minutes)

Quantum models are tuned and evaluated at 1000 shots (protocol of notebook 11); raw amplitude also with exact kernels (protocol of notebook 10).

In [ ]:
from sklearn.utils import resample
B = 1000
KEY_QUANTUM = ["amplitude, raw", "amplitude, min-max", "angle, robust + YJ"]
conf_all = np.arange(len(conf["y"]))
scores = {}          # model -> list of per-seed score arrays on the confirmation set

def tune_and_score(name, idx, y_tr, folds, shots, rng):
    family = QUANTUM[name][0]
    best = (-1, None, None, None)
    for f in (F_GRID if family == "1P1Q" else [None]):
        _, K_tr = kernel_builder(name, idx, f)
        K_fit = shot_noise(K_tr, shots, rng, symmetric=True) if shots else K_tr
        for C in C_GRID:
            sc = cv_auc(K_fit, y_tr, folds, C)
            if sc > best[0]:
                best = (sc, C, f, K_fit)
    _, C, f, K_fit = best
    K_eval, _ = kernel_builder(name, idx, f)
    K_c = K_eval(conf, conf_all)
    if shots:
        K_c = shot_noise(K_c, shots, rng)
    return SVC(kernel="precomputed", C=C, max_iter=MAX_ITER).fit(K_fit, y_tr).decision_function(K_c)

for s in SEEDS:
    idx = balanced_subsample(dev["y"], train, N_PER_CLASS, seed=s)
    y_tr = dev["y"][idx]
    folds = list(StratifiedKFold(5, shuffle=True, random_state=s).split(idx, y_tr))
    for name in KEY_QUANTUM:
        rng = np.random.default_rng(5000 + 17 * s + B)
        scores.setdefault(f"{name} ({B} shots)", []).append(tune_and_score(name, idx, y_tr, folds, B, rng))
    scores.setdefault("amplitude, raw (exact)", []).append(tune_and_score("amplitude, raw", idx, y_tr, folds, None, None))
    sc = StandardScaler().fit(dev["X"][idx])
    gs = GridSearchCV(SVC(kernel="rbf", max_iter=MAX_ITER), {"C": C_GRID[:7], "gamma": ["scale", 0.01, 0.1, 1.0]},
                      scoring="roc_auc", cv=folds).fit(sc.transform(dev["X"][idx]), y_tr)
    scores.setdefault("RBF-SVM (classical)", []).append(gs.best_estimator_.decision_function(sc.transform(conf["X"])))
    gs = GridSearchCV(HistGradientBoostingClassifier(random_state=s),
                      {"max_leaf_nodes": [7, 15, 31], "learning_rate": [0.03, 0.1], "max_iter": [100, 300]},
                      scoring="roc_auc", cv=folds).fit(dev["X"][idx], y_tr)
    scores.setdefault("BDT (classical)", []).append(gs.best_estimator_.predict_proba(conf["X"])[:, 1])
    print(f"seed {s} done", flush=True)

# references without training (identical for every seed)
valid = conf["qgl"] >= 0
scores["QG_axis2 alone (no training)"] = [-conf["X"][:, 2]] * len(SEEDS)     # smaller axis2 -> quark
scores["CMS QGL"] = [conf["qgl"]] * len(SEEDS)
np.savez_compressed(RESULTS / "confirmation_scores.npz", y=conf["y"], pt=conf["pt"], valid=valid,
                    **{k.replace(" ", "_").replace(",", "").replace("(", "").replace(")", ""): np.array(v) for k, v in scores.items()})
print(f"jets without QGL in the confirmation file: {int((~valid).sum())} (excluded everywhere below)")

## 3. Inclusive, per-bin and pT-reweighted AUC

pT reweighting: gluon jets get weights so that their pT spectrum (30 log-spaced bins) matches the quark spectrum. A model that only learned "high pT means quark" gets AUC 0.5 after reweighting.

In [ ]:
y_c, pt_c = conf["y"][valid], conf["pt"][valid]
edges_w = np.logspace(np.log10(30), np.log10(pt_c.max() * 1.001), 31)
hq, _ = np.histogram(pt_c[y_c == 1], edges_w, density=True)
hg, _ = np.histogram(pt_c[y_c == 0], edges_w, density=True)
bin_of = np.clip(np.digitize(pt_c, edges_w) - 1, 0, len(hq) - 1)
w = np.where(y_c == 1, 1.0, np.divide(hq, hg, out=np.zeros_like(hq), where=hg > 0)[bin_of])
print(f"check: AUC of pT itself, inclusive {roc_auc_score(y_c, pt_c):.3f}, reweighted {roc_auc_score(y_c, pt_c, sample_weight=w):.3f}")

pt_bins = {"30-100": (30, 100), "100-300": (100, 300), ">300": (300, np.inf)}
MODEL_ORDER = ["amplitude, min-max (1000 shots)", "angle, robust + YJ (1000 shots)", "amplitude, raw (1000 shots)",
               "amplitude, raw (exact)", "RBF-SVM (classical)", "BDT (classical)", "QG_axis2 alone (no training)", "CMS QGL"]

def auc_set(sc, mask=None, weights=None):
    s = np.asarray(sc)[valid]
    m = np.ones_like(y_c, bool) if mask is None else mask
    return roc_auc_score(y_c[m], s[m], sample_weight=None if weights is None else weights[m])

table = {}
print(f"{'model':34s} {'inclusive':>13s} {'pT-reweighted':>14s} " + " ".join(f"{b:>13s}" for b in pt_bins))
for name in MODEL_ORDER:
    row = {"inclusive": [auc_set(s) for s in scores[name]],
           "reweighted": [auc_set(s, weights=w) for s in scores[name]]}
    for b, (lo, hi) in pt_bins.items():
        row[b] = [auc_set(s, mask=(pt_c >= lo) & (pt_c < hi)) for s in scores[name]]
    table[name] = row
    fmt = lambda v: f"{np.mean(v):.3f} ± {np.std(v, ddof=1) if len(set(v)) > 1 else 0:.3f}"
    print(f"{name:34s} {fmt(row['inclusive']):>13s} {fmt(row['reweighted']):>14s} " + " ".join(f"{fmt(row[b]):>13s}" for b in pt_bins))
print("quark fraction per bin:", " ".join(f"{y_c[(pt_c >= lo) & (pt_c < hi)].mean():.3f}" for lo, hi in pt_bins.values()))

## 4. Paired bootstrap: AUC differences

For each bootstrap resample of confirmation jets, the AUC difference is averaged over the 5 seeds. Reported: mean difference and the central 68% interval (≈ ±1σ). 200 resamples.

In [ ]:
rng_b = np.random.default_rng(7)
n_jets = len(y_c)
boot_idx = [rng_b.integers(0, n_jets, n_jets) for _ in range(200)]

def paired(a, b, weights=None):
    diffs = []
    for bi in boot_idx:
        yy = y_c[bi]; ww = None if weights is None else weights[bi]
        d = [roc_auc_score(yy, np.asarray(sa)[valid][bi], sample_weight=ww) - roc_auc_score(yy, np.asarray(sb)[valid][bi], sample_weight=ww)
             for sa, sb in zip(scores[a], scores[b])]
        diffs.append(np.mean(d))
    lo, hi = np.percentile(diffs, [16, 84])
    return np.mean(diffs), lo, hi

COMPARE = [("amplitude, min-max (1000 shots)", "RBF-SVM (classical)"),
           ("amplitude, min-max (1000 shots)", "CMS QGL"),
           ("RBF-SVM (classical)", "CMS QGL"),
           ("amplitude, min-max (1000 shots)", "QG_axis2 alone (no training)"),
           ("amplitude, min-max (1000 shots)", "amplitude, raw (1000 shots)")]
paired_results = {}
for a, b in COMPARE:
    inc = paired(a, b); rw = paired(a, b, weights=w)
    paired_results[f"{a} - {b}"] = {"inclusive": inc, "reweighted": rw}
    print(f"{a} − {b}\n    inclusive {inc[0]:+.4f} [{inc[1]:+.4f}, {inc[2]:+.4f}]   pT-reweighted {rw[0]:+.4f} [{rw[1]:+.4f}, {rw[2]:+.4f}]")

json.dump({"event_overlap": len(overlap), "jets_without_qgl": int((~valid).sum()), "table": table, "paired": paired_results},
          open(RESULTS / "pt_binned_comparison.json", "w"), indent=2)
print("saved results/pt_binned_comparison.json")

## 5. Figure: pT-reweighted AUC (pT carries no information)

In [ ]:
COL = {"amplitude": COLORS["series1"], "angle": COLORS["series2"]}
fig, ax = plt.subplots(figsize=(8.5, 4.6))
ypos = np.arange(len(MODEL_ORDER))[::-1]
for yp, name in zip(ypos, MODEL_ORDER):
    col = COL.get(name.split(",")[0], COLORS["reference"])
    r = table[name]["reweighted"]; i = table[name]["inclusive"]
    ax.errorbar(np.mean(r), yp, xerr=np.std(r, ddof=1) if len(set(r)) > 1 else 0, fmt="o", color=col, markersize=8, markeredgecolor="white", capsize=3)
    ax.plot(np.mean(i), yp, "|", color=col, markersize=12, markeredgewidth=2)
ax.set_yticks(ypos, MODEL_ORDER)
ax.set_xlabel("AUC on the confirmation set (quark vs gluon)")
ax.set_title("pT-reweighted AUC (dot) and inclusive AUC (bar)", loc="left", fontsize=10)
ax.grid(axis="y", visible=False)
fig.tight_layout()
save(fig, "fig14_pt_reweighted")
plt.show()